# 04 · RadioVue single-X-ray research pipeline (end to end)

> **Research use only.** Generated radiographs and CT volumes are synthetic and patient-conditioned. They are
> estimates, not acquisitions of the patient and not clinical ground truth. Never use them for diagnosis or treatment.
> Upload only images you are authorised to use, and remove burned-in identifiers first.

**How to use:** `Runtime > Change runtime type > GPU (L4 recommended)`, then `Runtime > Run all`.

1. Setup installs everything and stages the models (first run ~15 min).
2. **Step 5** runs the public CC0 PA demo, so you can confirm the pipeline works before using your own image.
3. **Step 6 is where you upload your X-ray** (one DICOM, PNG or JPEG). Optionally declare its view first.
4. Step 7 shows the original vs the SYNTHETIC view, the uncertainty map, QC, anatomy and the CT.

Everything goes through one function: `run_pipeline(input_xray)`. Outputs and provenance are saved to Drive under
`OUTPUT_ROOT`. A synthetic view that fails QC is **never** passed to Volumea.

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/sanafathima-00/RadioVue.git"  #@param {type:"string"}
REPO_BRANCH = "claude/single-xray-vlm-pipeline"  #@param {type:"string"}
#@markdown If the pipeline code is not pushed yet, upload `radiovue_colab_bundle.zip` (see SETUP_GUIDE.md) to this Drive path:
CODE_BUNDLE_ON_DRIVE = "RadioVue/code/radiovue_colab_bundle.zip"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/RadioVue"  #@param {type:"string"}
DVG_CHECKPOINTS_ON_DRIVE = DRIVE_ROOT + "/Volumea/models/DVG-Diffusion"
PARALLAX_MODEL_ON_DRIVE = DRIVE_ROOT + "/Parallax/models/DiT-fb-512"  # used if present, else downloaded from Hugging Face
OUTPUT_ROOT = DRIVE_ROOT + "/outputs/single_xray_pipeline"
REPO_DIR = "/content/RadioVue"

In [ ]:
#@title 2. GPU, Drive and code
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout[:700] or smi.stderr)
assert smi.returncode == 0, "No GPU. Runtime > Change runtime type > GPU (L4/A100 recommended, T4 minimum)."

from google.colab import drive
drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive") / CODE_BUNDLE_ON_DRIVE
if not Path(REPO_DIR, "radiovue_pipeline").exists():
    if bundle.is_file():
        Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(bundle).extractall(REPO_DIR)
        print("Code from Drive bundle:", bundle)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
assert Path(REPO_DIR, "radiovue_pipeline").exists(), (
    "radiovue_pipeline/ not found. Push the branch that contains it, or upload the code bundle (SETUP_GUIDE.md).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code revision:", rev or "(bundle, see BUNDLE_MANIFEST.json)")

# colab_setup has no third-party imports, so it can run before packages are installed.
spec = importlib.util.spec_from_file_location("colab_setup", f"{REPO_DIR}/radiovue_pipeline/colab_setup.py")
cs = importlib.util.module_from_spec(spec); spec.loader.exec_module(cs)
GPU = cs.gpu_report()

In [ ]:
#@title 3. Install dependencies and stage models (first run ~10-15 min)
cs.install_runtime(parallax=True, dvg=True, tools=True)
cs.build_drr_projector(f"{REPO_DIR}/Volumea")
DVG_CKPT_DIR = cs.stage_dvg_checkpoints(DVG_CHECKPOINTS_ON_DRIVE, verify_hash=False)
cs.ensure_parallax_shims(f"{REPO_DIR}/Parallax")
PARALLAX_MODEL_DIR = cs.stage_parallax_model(PARALLAX_MODEL_ON_DRIVE)
HF_TOKEN = cs.hf_token_from_colab_secrets()  # value is never printed
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access."
cs.check_gated_access("google/medgemma-1.5-4b-it")
MEDGEMMA_PYTHON = cs.make_medgemma_venv("/content/medgemma_venv")

In [ ]:
#@title 4. Build the pipeline (loads calibration from notebook 03 if it exists)
import json
from radiovue_pipeline import PipelineConfig, run_pipeline
from radiovue_pipeline.pipeline import Components
cfg = PipelineConfig(output_root=OUTPUT_ROOT, volumea_root=f"{REPO_DIR}/Volumea", parallax_root=f"{REPO_DIR}/Parallax",
                     dvg_checkpoint_dir=DVG_CKPT_DIR, parallax_model_dir=PARALLAX_MODEL_DIR)
calib_path = Path(OUTPUT_ROOT) / "calibration" / "geometry_calibration.json"
if calib_path.exists():
    calib = json.loads(calib_path.read_text())
    for key in ("lateral_azimuth_deg", "lateral_azimuth_calibrated", "parallax_output_transform",
                "dvg_frontal_transform", "dvg_lateral_transform", "dvg_geometry_calibrated"):
        setattr(cfg, key, calib[key])
    print("Geometry calibration loaded:", {k: calib[k] for k in calib if k != "evidence"})
else:
    print("WARNING: no geometry calibration found. Lateral sign and DVG orientation are UNCALIBRATED (run notebook 03).")
comps = Components.load(cfg, medgemma_python=MEDGEMMA_PYTHON, repo_root=REPO_DIR)
print("Load errors so far:", comps.load_errors or "none")

In [ ]:
#@title Display helper
import matplotlib.pyplot as plt
from PIL import Image
def show(result):
    print("STATUS:", result["status"], "| input view:", result["input_view"], "| generated:", result["generated_view_type"])
    for w in result["warnings"]: print("  !", w)
    panels = [("ORIGINAL (rendered)", result["original_image"]["rendered_grayscale"])]
    if result["generated_view"]:
        panels += [("SYNTHETIC view (seed 0)", result["generated_view"]["display_path"]),
                   ("Uncertainty: per-pixel std over seeds", result["view_synthesis_metadata"]["uncertainty_map"])]
    seg = (result.get("anatomical_extraction") or {}).get("segmentation") or {}
    if seg.get("overlay_path"): panels.append(("Segmentation (PSPNet)", seg["overlay_path"]))
    if result["ct_reconstruction"]:
        panels += [(f"SYNTHETIC CT {k[-5:]}", result["ct_reconstruction"][k]) for k in ("preview_axis0", "preview_axis1", "preview_axis2")]
    fig, ax = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    for a, (t, p) in zip(np.atleast_1d(ax), panels): a.imshow(Image.open(p), cmap="gray"); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.show()
    qc = result["validation_results"].get("synthesis_qc")
    if qc: print("QC passed:", qc["passed"], json.dumps({k: {kk: vv for kk, vv in v.items() if kk in ("passed", "gating", "row_profile_corr", "mean_pixel_std", "input_output_ncc", "p99_minus_p1")} for k, v in qc["checks"].items()}, indent=1))
    ext = result.get("anatomical_extraction") or {}
    if ext.get("image_metadata"): print("MedGemma view:", ext["image_metadata"])
    print("Run folder:", result["run_dir"])
import numpy as np

In [ ]:
#@title 5. Verification run on the public CC0 PA (declared PA because PNG has no view metadata)
import hashlib, urllib.request
PUBLIC = {
    "pa_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png",
                        "a43431b089a6dfab031c10eceef6c821e4f326913bf9015f73f4a1f0446d2be1"),
    "lateral_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/9/9d/Chest_Xray_Lateral_3-8-2010.png",
                             "d07a306fb285a8456739e0349fb208b64ade12eb4bb84bc09889c7b3ec7f99a7"),
}
PUBLIC_DIR = Path("/content/public_cc0"); PUBLIC_DIR.mkdir(exist_ok=True)
for name, (url, digest) in PUBLIC.items():
    path = PUBLIC_DIR / name
    if not path.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "RadioVueResearch/0.1"})
        path.write_bytes(urllib.request.urlopen(req).read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Checksum mismatch: {name}"
print("Public test images verified:", sorted(p.name for p in PUBLIC_DIR.iterdir()))

demo = run_pipeline(PUBLIC_DIR / "pa_cc0_2010.png", cfg, comps, declared_view="PA")
show(demo)

## 6 · Upload your X-ray here
Choose the view only if you know it from the acquisition. Leave it as `auto` otherwise. AP is never relabelled PA.
A lateral input stops before synthesis by default, because no validated lateral-to-frontal generator exists.

In [ ]:
#@title 6. Upload ONE chest X-ray (DICOM / PNG / JPEG)
DECLARED_VIEW = "auto"  #@param ["auto", "PA", "AP", "LATERAL"]
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one file."
name = next(iter(uploaded))
UPLOAD_DIR = Path("/content/user_upload"); UPLOAD_DIR.mkdir(exist_ok=True)
user_path = UPLOAD_DIR / ("user_input" + Path(name).suffix.lower())
user_path.write_bytes(uploaded[name]); del uploaded
print("Stored as", user_path.name, f"({user_path.stat().st_size} bytes). Original filename is not recorded in outputs.")

In [ ]:
#@title 7. Run the pipeline on your X-ray
result = run_pipeline(user_path, cfg, comps, declared_view=None if DECLARED_VIEW == "auto" else DECLARED_VIEW)
show(result)

In [ ]:
#@title 8. Inspect the CT volume interactively (normalised DVG units, not HU)
import ipywidgets as W
if result["ct_reconstruction"]:
    vol = np.load(result["ct_reconstruction"]["npy"])
    def view(axis=0, index=64):
        sl = np.take(vol, index, axis=axis)
        plt.figure(figsize=(5, 5)); plt.imshow(sl, cmap="gray", vmin=0, vmax=1); plt.title(f"SYNTHETIC CT axis {axis} slice {index}"); plt.axis("off"); plt.show()
    W.interact(view, axis=W.IntSlider(0, 0, 2), index=W.IntSlider(64, 0, 127))
else:
    print("No CT was produced:", result["status"])